In [1]:
pip install pandas scikit-learn xgboost

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Program Files\Python39\python.exe -m pip install --upgrade pip' command.


In [2]:
import pandas as pd

train = pd.read_csv("data/transformed_train.csv")
val = pd.read_csv("data/transformed_val.csv")
test = pd.read_csv("data/transformed_test.csv")

In [3]:
X_train = train.drop("Demand", axis=1)
y_train = train["Demand"]

X_val = val.drop("Demand", axis=1)
y_val = val["Demand"]

X_test = test.drop("Demand", axis=1)
y_test = test["Demand"]

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

y_val_pred = rf.predict(X_val)



In [7]:
mse = mean_squared_error(y_val, y_val_pred)
r2 = r2_score(y_val, y_val_pred)
print(f"Validation MSE: {mse:.4f}")
print(f"Validation R^2: {r2:.4f}")


Validation MSE: 595.6243
Validation R^2: 0.7436


In [16]:
!pip uninstall xgboost -y
!pip install xgboost==2.0.3

Found existing installation: xgboost 2.1.4
Uninstalling xgboost-2.1.4:
  Successfully uninstalled xgboost-2.1.4


ERROR: Exception:
Traceback (most recent call last):
  File "c:\program files\python39\lib\site-packages\pip\_internal\cli\base_command.py", line 180, in _main
    status = self.run(options, args)
  File "c:\program files\python39\lib\site-packages\pip\_internal\commands\uninstall.py", line 89, in run
    uninstall_pathset.commit()
  File "c:\program files\python39\lib\site-packages\pip\_internal\req\req_uninstall.py", line 442, in commit
    self._moved_paths.commit()
  File "c:\program files\python39\lib\site-packages\pip\_internal\req\req_uninstall.py", line 282, in commit
    save_dir.cleanup()
  File "c:\program files\python39\lib\site-packages\pip\_internal\utils\temp_dir.py", line 184, in cleanup
    rmtree(self._path)
  File "c:\program files\python39\lib\site-packages\pip\_vendor\tenacity\__init__.py", line 341, in wrapped_f
    return self(f, *args, **kw)
  File "c:\program files\python39\lib\site-packages\pip\_vendor\tenacity\__init__.py", line 432, in __call__
    do = self

Defaulting to user installation because normal site-packages is not writeable

You should consider upgrading via the 'c:\program files\python39\python.exe -m pip install --upgrade pip' command.


In [19]:
import xgboost
print(xgboost.__version__)
print(xgboost.__file__)

2.1.4
C:\Users\Shambhavi Chaturvedi\AppData\Roaming\Python\Python39\site-packages\xgboost\__init__.py


In [23]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    tree_method="hist",
    random_state=42,
    early_stopping_rounds=20
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)
y_val_pred = xgb.predict(X_val)

mse = mean_squared_error(y_val, y_val_pred)
r2 = r2_score(y_val, y_val_pred)
print(f"Validation MSE: {mse:.4f}")
print(f"Validation R^2: {r2:.4f}")




Validation MSE: 673.6658
Validation R^2: 0.7100


In [26]:
import pandas as pd
import numpy as np

def forecast_ensemble_recursive(rf_model, xgb_model, df, T, feature_cols, target_col="Demand"):
    
    df = df.copy()
    predictions = []

    # Keep demand history
    demand_history = list(df[target_col].values)

    for step in range(T):
        
        new_row = df.iloc[-1].copy()

        # 🔮 Predict using BOTH models
        X_input = new_row[feature_cols].values.reshape(1, -1)
        
        pred_rf = rf_model.predict(X_input)[0]
        pred_xgb = xgb_model.predict(X_input)[0]

        # 🎯 Ensemble (45/55)
        pred = 0.45 * pred_rf + 0.55 * pred_xgb
        predictions.append(pred)

        # Update history
        demand_history.append(pred)
        new_row[target_col] = pred

        # 🔁 Demand lags (CORRECT)
        new_row["lag_demand_1"] = demand_history[-1]
        new_row["lag_demand_7"] = demand_history[-7] if len(demand_history) >= 7 else demand_history[-1]
        new_row["lag_demand_28"] = demand_history[-28] if len(demand_history) >= 28 else demand_history[-1]

        # 📊 Rolling features
        new_row["roll_demand_mean_7"] = np.mean(demand_history[-7:])
        new_row["roll_demand_std_7"] = np.std(demand_history[-7:])
        new_row["roll_demand_mean_28"] = np.mean(demand_history[-28:])

        # ⏱️ Time index
        new_row["time_idx"] += 1

        # 🗓️ Day-of-week cyclic update (IMPORTANT)
        dow = (step + 1) % 7
        new_row["dow_sin"] = np.sin(2 * np.pi * dow / 7)
        new_row["dow_cos"] = np.cos(2 * np.pi * dow / 7)

        # 🔄 Keep other lag features stable (for now)
        # (inventory, price, etc. — unless you simulate them)

        # Append
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

    return predictions

In [27]:
T = 14  # predict next 14 days

feature_cols = X_train.columns.tolist()

preds = forecast_recursive(
    model=rf,   # or xgb
    df=train,
    T=T,
    feature_cols=feature_cols
)

print(preds)

C:\Users\Shambhavi Chaturvedi\AppData\Roaming\Python\Python39\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Shambhavi Chaturvedi\AppData\Roaming\Python\Python39\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Shambhavi Chaturvedi\AppData\Roaming\Python\Python39\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Shambhavi Chaturvedi\AppData\Roaming\Python\Python39\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Shambhavi Chaturvedi\AppData\Roaming\Python\Python39\site-p

[np.float64(128.87398651852962), np.float64(129.05642028013287), np.float64(128.63659985661158), np.float64(128.68459985661156), np.float64(127.99070328279682), np.float64(127.783078778362), np.float64(120.24243471577034), np.float64(120.35100936184499), np.float64(119.99644127201604), np.float64(119.83892946541522), np.float64(119.53303276714783), np.float64(120.16578643934264), np.float64(119.86881229317152), np.float64(119.27385361918357)]
